# Pipeline Module Tests

This notebook tests **each part of the pipeline individually**. Use it to verify that every module works in isolation.

The other notebook (`vggt_coliseum_inpaint_pipeline.ipynb`) runs the **full end-to-end pipeline** — this one is for unit-style testing of each component.

**Modules tested:**
1. Config — PipelineConfig, RenderResult, InpaintResult
2. IO — list_data_images, load/save, video extraction
3. Geometry — camera math, pose comparison
4. Point cloud + Rendering — synthetic points → render
5. VGGT — load model, run reconstruction (needs `data/`)
6. Orbit — estimate orbit, camera_at_angle, pick_next_angles
7. Inpainting — hole mask, OpenCV fallback
8. Metrics — LPIPS
9. Viz — plot_point_cloud_3d, plot_image_grid

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import utils

plt.rcParams['figure.dpi'] = 120
out_dir = utils.ensure_dir("outputs/notebook_module_tests")

: 

## 1. Config

In [ ]:
cfg = utils.PipelineConfig(data_dir="data", vggt_conf_percentile=55.0)
print("PipelineConfig:", cfg.vggt_model_id, cfg.inpaint_model_id)

# RenderResult / InpaintResult are created by pipeline functions, not manually
print("Config OK")

## 2. IO

In [ ]:
images = utils.list_data_images("data")
print("list_data_images:", [p.name for p in images])

videos = utils.list_videos("data")
print("list_videos:", [p.name for p in videos])

if images:
    img = utils.load_pil_rgb(images[0])
    test_path = out_dir / "io_test_save.png"
    utils.save_pil(img, test_path)
    print("load_pil_rgb + save_pil OK")
else:
    print("No images in data/ — skip load/save test")

if videos:
    frame = utils.extract_frame_from_video(videos[0], frame_index="middle")
    print("extract_frame_from_video OK, size:", frame.size)

## 3. Geometry

In [ ]:
extr = np.eye(4, dtype=np.float32)
extr[:3, 3] = [1, 0, 0]
center = utils.camera_center_from_extrinsic(extr)
print("camera_center_from_extrinsic:", center)

novel = utils.perturb_camera_extrinsic(extr, shift_right=0.2, yaw_deg=-5)
print("perturb_camera_extrinsic OK, shape:", novel.shape)

pts = np.array([[1, 0, 2], [0, 1, 2]], dtype=np.float32)
intr = np.array([[200, 0, 128], [0, 200, 128], [0, 0, 1]], dtype=np.float32)
uv, z, valid = utils.project_world_points(pts, extr, intr)
print("project_world_points:", uv.shape, "valid:", valid)
print("Geometry OK")

## 4. Point cloud + Rendering (synthetic)

In [ ]:
utils.seed_everything(0)
n = 2000
rng = np.random.default_rng(0)
pts = np.stack([rng.uniform(-1, 1, n), rng.uniform(-0.5, 0.5, n), rng.uniform(2, 4, n)], axis=1).astype(np.float32)
cols = np.clip(pts - pts.min(axis=0), 0, 1).astype(np.float32)

h, w = 256, 256
intr = np.array([[180, 0, w/2], [0, 180, h/2], [0, 0, 1]], dtype=np.float32)
extr = np.concatenate([np.eye(3), np.zeros((3, 1))], axis=1).astype(np.float32)

render = utils.render_projected_point_cloud(pts, cols, extr, intr, (h, w), point_radius=1)
print("render_projected_point_cloud:", render.projected_count, "points, valid_ratio:", f"{render.valid_mask.mean():.4f}")
display(render.image)
utils.save_pil(render.image, out_dir / "04_synthetic_render.png")
print("Rendering OK")

## 5. VGGT (needs data/ with images)

In [ ]:
images = utils.list_data_images("data")
if len(images) < 2:
    print("Need at least 2 images in data/ — skip VGGT test")
    scene = None
else:
    paths = [utils.ensure_png_copy(p) for p in images[:2]]
    device = utils.get_device()
    model = utils.load_vggt_model(device=device)
    scene = utils.run_vggt_reconstruction(paths, model=model, device=device)
    del model
    print("VGGT scene:", utils.describe_scene(scene))
    print("VGGT OK")

## 6. Orbit (uses scene from VGGT or synthetic extrinsics)

In [ ]:
if scene is not None:
    extrinsics = [np.asarray(scene["extrinsic"][i]) for i in range(len(scene["extrinsic"]))]
    pts_merged, _ = utils.merge_scene_point_cloud(scene, conf_percentile=55, max_points=10000)
    centroid = pts_merged.mean(axis=0)
    orbit, cam_height = utils.estimate_horizontal_orbit(extrinsics, centroid, point_cloud=pts_merged)
    print("estimate_horizontal_orbit: radius=", orbit.radius, "gap_deg=", np.degrees(orbit.gap_size))
    angles = utils.pick_next_angles_bilateral(orbit, step_deg=20)
    print("pick_next_angles_bilateral:", [np.degrees(a) for a in angles])
    if angles:
        novel_extr, novel_intr = utils.camera_at_angle(orbit, angles[0], cam_height, scene["intrinsic"][0], extrinsics)
        print("camera_at_angle OK")
    print("Orbit OK")
else:
    print("No scene — skip orbit test")

## 7. Inpainting (hole mask + OpenCV fallback)

In [ ]:
novel_extr = utils.perturb_camera_extrinsic(extr, shift_right=0.2, yaw_deg=-8)
novel_render = utils.render_projected_point_cloud(pts, cols, novel_extr, intr, (h, w), point_radius=1)
novel_img, hole_mask, overlay = utils.prepare_novel_view_inpainting_inputs(novel_render, dilate_px=3, close_px=2)
print("prepare_novel_view_inpainting_inputs: hole_ratio=", f"{(np.asarray(hole_mask) > 0).mean():.4f}")

inpainted = utils.opencv_inpaint_fallback(novel_img, hole_mask, radius=5)
display(utils.plot_image_grid([novel_img, hole_mask.convert("RGB"), inpainted], ["Sparse", "Hole mask", "OpenCV inpainted"])[0])
utils.save_pil(inpainted, out_dir / "07_opencv_inpaint.png")
print("Inpainting OK")

## 8. Metrics (LPIPS)

In [ ]:
device = utils.get_device()
metrics = utils.compute_image_metrics(render.image, inpainted, lpips_device=device)
print("compute_image_metrics:", metrics)
print("Metrics OK")

## 9. Viz

In [ ]:
fig, ax = utils.plot_point_cloud_3d(pts, cols, title="Synthetic point cloud", point_size=0.3)
plt.show()
utils.save_matplotlib_figure(fig, out_dir / "09_point_cloud.png")

fig2, _ = utils.plot_image_grid([render.image, novel_render.image, inpainted], ["Base", "Novel", "Inpainted"])
plt.show()
utils.save_matplotlib_figure(fig2, out_dir / "09_grid.png")
print("Viz OK")
print("All module tests done. Outputs in:", out_dir.resolve())